# cbj · 공통 앵커 준비와 상세 EDA
**전체 유전자 → 변이 유형 → 위치·아미노산 → 기능 단계**를 하나의 표현으로 연결합니다. 기능 단계는 관측 가능한 21개 유전자의 보조 블록이며 생체 흐름·메틸화량을 추정하지 않습니다.

전체 원본 읽기와 계산은 Kaggle에서만 실행합니다. 이 노트북은 모델을 학습하지 않습니다. 파싱·그룹 분할·fold별 피처 변환·새 표기에 대한 정보 보존을 검증하고 학습 직전 입력을 저장합니다.

새로운 변이에 상위 표현이 남는 것은 **표현의 대응 가능성**입니다. 새 데이터의 예측 정확도 향상은 이후 OOF/외부 검증 전에는 미확인입니다. test는 라벨 없는 스키마·결측·미지 패턴 감사와 고정 변환에만 사용하며 vocabulary 선정에 쓰지 않습니다.

In [ ]:
from pathlib import Path
import os,sys,json,gzip,hashlib,itertools,copy,unittest,time
assert Path('/kaggle/input').exists(), 'Kaggle-only full-data preparation'
os.environ['MPLCONFIGDIR']='/tmp/cbj_anchor_mpl'
OUT=Path('/kaggle/working/cbj_anchor_ready');OUT.mkdir(exist_ok=True)
Path('/kaggle/working/cbj_anchor_features.py').write_text('"""Versioned, label-independent anchor representation. No classifier fitting here."""\nfrom collections import Counter\nimport hashlib\nimport json\nimport math\nimport re\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix\n\nVERSION=\'cbj-anchor-features-v2\'\nAA=\'ACDEFGHIKLMNPQRSTVWY\'\nTYPES=(\'synonymous\',\'missense\',\'stop_gained\',\'stop_lost\',\'frameshift\',\'deletion\',\'insertion\',\'delins\',\'complex\',\'unparsed\')\nMISSING={\'\',\'NA\',\'N/A\',\'NAN\',\'NULL\',\'NONE\',\'<NA>\',\'.\'}\n\n\ndef parse_token(token):\n    t=token.removeprefix(\'p.\')\n    d={\'normalized_token\':t,\'type\':\'unparsed\',\'site\':\'\',\'position_start\':None,\'position_end\':None,\n       \'aa_from\':\'\',\'aa_to\':\'\',\'change\':\'\',\'parse_status\':\'unresolved\'}\n    m=re.fullmatch(r\'([A-Z*])(\\d+)([A-Z*=])\',t)\n    if m and m[1] in AA+\'*\' and m[3] in AA+\'*=\':\n        a,p,b=m.groups();b=a if b==\'=\' else b\n        typ=\'synonymous\' if a==b else (\'stop_gained\' if b==\'*\' else (\'stop_lost\' if a==\'*\' else \'missense\'))\n        d.update(type=typ,site=str(int(p)),position_start=int(p),position_end=int(p),aa_from=a,aa_to=b,\n                 change=f\'{a}>{b}\',parse_status=\'simple_protein_notation\')\n    elif re.fullmatch(r\'[A-Z]+\\d+(?:[A-Z])?fs(?:\\*\\d+|Ter\\d+)?\',t):\n        d.update(type=\'frameshift\',parse_status=\'type_only_ambiguous_position\')\n        m=re.match(r\'([A-Z]+)(\\d+)\',t)\n        if len(m[1])==1 and m[1] in AA:\n            d.update(site=str(int(m[2])),position_start=int(m[2]),aa_from=m[1],parse_status=\'frameshift_start_only\')\n    else:\n        m=re.fullmatch(r\'([A-Z]?)(\\d+)(?:_([A-Z]?)(\\d+))?(delins|del|ins)([A-Z]*)\',t)\n        if m:\n            a,p,a2,q,kind,alt=m.groups()\n            if (not a or a in AA) and (not a2 or a2 in AA) and all(c in AA for c in alt):\n                start=int(p);end=int(q or p)\n                if end>=start and (kind==\'del\' or alt):\n                    d.update(type={\'del\':\'deletion\',\'ins\':\'insertion\',\'delins\':\'delins\'}[kind],\n                             site=str(start) if start==end else f\'{start}_{end}\',position_start=start,position_end=end,\n                             aa_from=a,aa_to=alt,parse_status=\'indel_notation_coordinates_only\')\n        elif re.fullmatch(r\'\\d+_\\d+[A-Z]+>[A-Z]+\',t):\n            m=re.fullmatch(r\'(\\d+)_(\\d+)([A-Z]+)>([A-Z]+)\',t)\n            if int(m[2])>=int(m[1]) and all(c in AA for c in m[3]+m[4]):\n                d.update(type=\'complex\',site=f\'{int(m[1])}_{int(m[2])}\',position_start=int(m[1]),position_end=int(m[2]),\n                         aa_from=m[3],aa_to=m[4],parse_status=\'range_notation_not_synonymous_assumed\')\n    return d\n\n\ndef build_samples(df,genes,stage_genes):\n    """Preserve raw annotations; prediction features ignore token order/repetition.\n\n    Missing columns become unknown observations, never WT. Unexpected columns fail.\n    No mutation severity score, cross-gene position magnitude, labels, or learned cutoffs.\n    """\n    extras=set(df.columns)-set(genes)-{\'ID\',\'SUBCLASS\'}\n    if extras:raise ValueError(f\'Unrecognized gene/schema columns: {sorted(extras)[:8]}\')\n    if \'ID\' not in df or not df.ID.is_unique:raise ValueError(\'Unique sample ID required\')\n    n=len(df);samples=[{\'ID\':str(x),\'features\':{},\'exact\':set(),\'site\':set(),\'gene_change\':set(),\n                      \'missing_genes\':set(),\'gene_types\':{},\'gene_tokens\':{},\'aa_counts\':Counter(),\n                      \'canonical_parts\':[],\'raw_hasher\':hashlib.sha256(),\'repeat_tokens\':0,\'token_count\':0} for x in df.ID]\n    annotations=[];missing_rows=[]\n    for gene in genes:\n        values=df[gene].tolist() if gene in df else [\'\']*n\n        for i,v in enumerate(values):\n            raw=\'\' if pd.isna(v) else str(v);value=raw.strip();s=samples[i]\n            raw_bytes=raw.encode();s[\'raw_hasher\'].update(len(raw_bytes).to_bytes(4,\'big\')+raw_bytes)\n            if value.upper() in MISSING:\n                s[\'missing_genes\'].add(gene);s[\'features\'][f\'gene|{gene}|missing\']=1.\n                s[\'canonical_parts\'].append(gene+\':<MISSING>\')\n                missing_rows.append({\'ID\':s[\'ID\'],\'gene\':gene,\'raw\':raw,\'column_absent\':gene not in df});continue\n            if value.upper()==\'WT\':continue\n            # Relationship-bearing delimiters are not silently treated as unordered variants.\n            tokens=[value] if any(c in value for c in \'[];,|\') else value.split()\n            # A nonempty punctuation-only value remains one unresolved token.\n            if not tokens:tokens=[value]\n            parsed=[parse_token(t) for t in tokens]\n            unique={p[\'normalized_token\']:p for p in parsed}\n            s[\'repeat_tokens\']+=len(tokens)-len(unique);s[\'token_count\']+=len(tokens)\n            s[\'gene_tokens\'][gene]=set(unique);s[\'gene_types\'][gene]={p[\'type\'] for p in unique.values()}\n            s[\'features\'][f\'gene|{gene}|nonWT\']=1.\n            s[\'canonical_parts\'].append(gene+\':\'+json.dumps(sorted(unique),ensure_ascii=True))\n            for k,(token,p) in enumerate(zip(tokens,parsed)):\n                annotations.append({\'ID\':s[\'ID\'],\'gene\':gene,\'raw\':raw,\'token\':token,\'token_index\':k+1,\n                                    \'tokens_in_cell\':len(tokens),**p})\n            for t,p in unique.items():\n                s[\'features\'][f\'type|{gene}|{p["type"]}\']=1.\n                s[\'exact\'].add(f\'exact|{gene}|{t}\')\n                if p[\'site\']:s[\'site\'].add(f\'site|{gene}|{p["site"]}\')\n                if p[\'change\'] and p[\'aa_from\'] in AA and p[\'aa_to\'] in AA:\n                    s[\'gene_change\'].add(f\'gene_change|{gene}|{p["change"]}\')\n                    s[\'aa_counts\'][p[\'change\']]+=1\n    for s in samples:\n        f=s[\'features\'];ng=len(s[\'gene_tokens\']);nt=sum(len(t) for t in s[\'gene_tokens\'].values())\n        s[\'mutated_gene_count\']=ng;s[\'missing_gene_count\']=len(s[\'missing_genes\']);s[\'unique_token_count\']=nt\n        f[\'global|burden_log\']=math.log1p(ng)/math.log1p(max(len(genes),1))\n        f[\'global|missing_fraction\']=len(s[\'missing_genes\'])/max(len(genes),1)\n        f[\'global|multiple_fraction\']=sum(len(t)>1 for t in s[\'gene_tokens\'].values())/max(ng,1)\n        for typ in TYPES:\n            f[f\'global|type_fraction|{typ}\']=sum(typ in t for t in s[\'gene_types\'].values())/max(ng,1)\n        for change,count in s[\'aa_counts\'].items():f[\'aa|\'+change]=count/max(sum(s[\'aa_counts\'].values()),1)\n        for change,count in s[\'aa_counts\'].items():\n            a,b=change.split(\'>\');den=max(sum(s[\'aa_counts\'].values()),1)\n            f[\'aa_from|\'+a]=f.get(\'aa_from|\'+a,0)+count/den\n            f[\'aa_to|\'+b]=f.get(\'aa_to|\'+b,0)+count/den\n        for stage,gs in stage_genes.items():\n            gs=[g for g in gs if g in genes]\n            present=[g for g in gs if g in s[\'gene_tokens\']]\n            missing=sum(g in s[\'missing_genes\'] for g in gs)\n            known=len(gs)-missing\n            if present:f[f\'path|{stage}|any\']=1.\n            elif missing or not gs:f[f\'path|{stage}|unknown\']=1.\n            f[f\'path|{stage}|missing_fraction\']=missing/max(len(gs),1)\n            f[f\'path|{stage}|gene_fraction\']=len(present)/max(known,1)\n            for typ in TYPES:f[f\'path|{stage}|type_fraction|{typ}\']=sum(typ in s[\'gene_types\'][g] for g in present)/max(known,1)\n        s[\'canonical_hash\']=hashlib.sha256(\'\\n\'.join(s.pop(\'canonical_parts\')).encode()).hexdigest()\n        s[\'raw_hash\']=s.pop(\'raw_hasher\').hexdigest()\n        for category in [\'exact\',\'site\',\'gene_change\']:\n            for key in s[category]:f[key]=1.\n        s[\'features\']={k:float(v) for k,v in f.items() if v!=0}\n    columns=[\'ID\',\'gene\',\'raw\',\'token\',\'token_index\',\'tokens_in_cell\',\'normalized_token\',\'type\',\'site\',\'position_start\',\'position_end\',\'aa_from\',\'aa_to\',\'change\',\'parse_status\']\n    return samples,pd.DataFrame(annotations,columns=columns),pd.DataFrame(missing_rows,columns=[\'ID\',\'gene\',\'raw\',\'column_absent\'])\n\n\nclass AnchorEncoder:\n    """Fixed semantic coarse features + fold-only vocabulary of finer patterns."""\n    def __init__(self,genes,stage_genes,min_support=3):\n        self.genes=list(genes);self.stage_genes=stage_genes;self.min_support=min_support\n\n    def fit(self,samples):\n        base=[]\n        for g in self.genes:\n            base.extend([f\'gene|{g}|nonWT\',f\'gene|{g}|missing\'])\n            base.extend(f\'type|{g}|{t}\' for t in TYPES)\n        base.extend(\'aa|\'+a+\'>\'+b for a in AA for b in AA)\n        base.extend(\'aa_from|\'+a for a in AA);base.extend(\'aa_to|\'+a for a in AA)\n        base.extend([\'global|burden_log\',\'global|missing_fraction\',\'global|multiple_fraction\'])\n        base.extend(\'global|type_fraction|\'+t for t in TYPES)\n        for stage in self.stage_genes:\n            base.extend(f\'path|{stage}|{x}\' for x in [\'any\',\'unknown\',\'missing_fraction\',\'gene_fraction\'])\n            base.extend(f\'path|{stage}|type_fraction|{t}\' for t in TYPES)\n        support=Counter();groups=set();self.observed_support=Counter();self.seen={k:set() for k in [\'exact\',\'site\',\'gene_change\']}\n        for s in samples:\n            for k in self.seen:self.seen[k].update(s[k])\n            if s[\'canonical_hash\'] not in groups:\n                support.update(set().union(*(s[k] for k in self.seen)));self.observed_support.update(s[\'features\'].keys());groups.add(s[\'canonical_hash\'])\n        self.names=base+sorted(k for k,n in support.items() if n>=self.min_support)\n        if len(self.names)!=len(set(self.names)):raise AssertionError(\'Duplicate feature names\')\n        self.index={k:i for i,k in enumerate(self.names)}\n        self.fit_ids=sorted(s[\'ID\'] for s in samples)\n        self.fit_group_count=len(groups)\n        return self\n\n    def transform(self,samples):\n        rr=[];cc=[];vv=[]\n        for i,s in enumerate(samples):\n            for k,v in s[\'features\'].items():\n                if k in self.index:rr.append(i);cc.append(self.index[k]);vv.append(v)\n        x=csr_matrix((np.array(vv,dtype=np.float32),(rr,cc)),shape=(len(samples),len(self.names)))\n        if not np.isfinite(x.data).all():raise ValueError(\'Nonfinite matrix\')\n        return x\n\n    def to_dict(self):\n        return {\'version\':VERSION,\'genes\':self.genes,\'stage_genes\':self.stage_genes,\'min_support\':self.min_support,\n                \'names\':self.names,\'seen\':{k:sorted(v) for k,v in self.seen.items()},\'fit_ids\':self.fit_ids,\'fit_group_count\':self.fit_group_count,\'observed_support\':dict(self.observed_support)}\n\n    @classmethod\n    def from_dict(cls,d):\n        if d[\'version\']!=VERSION:raise ValueError(\'Feature version mismatch\')\n        e=cls(d[\'genes\'],d[\'stage_genes\'],d[\'min_support\']);e.names=d[\'names\'];e.index={k:i for i,k in enumerate(e.names)}\n        e.seen={k:set(v) for k,v in d[\'seen\'].items()};e.fit_ids=d[\'fit_ids\'];e.fit_group_count=d[\'fit_group_count\'];e.observed_support=Counter(d[\'observed_support\']);return e\n\n    def novelty(self,samples):\n        rows=[]\n        for s in samples:\n            d={\'ID\':s[\'ID\'],\'canonical_hash\':s[\'canonical_hash\'],\'mutated_gene_count\':s[\'mutated_gene_count\']}\n            for k in self.seen:\n                d[k+\'_n\']=len(s[k]);d[k+\'_unseen\']=len(s[k]-self.seen[k])\n                d[k+\'_filtered_seen\']=sum(v in self.seen[k] and v not in self.index for v in s[k])\n                d[k+\'_retained\']=sum(v in self.index for v in s[k])\n            d[\'coarse_nonzero\']=sum(k in self.index and k.startswith((\'gene|\',\'type|\',\'aa|\')) for k in s[\'features\'])\n            d[\'unseen_exact_fraction\']=d[\'exact_unseen\']/d[\'exact_n\'] if d[\'exact_n\'] else np.nan\n            for key in [\'novel_tokens\',\'novel_parsed_type_n\',\'novel_aa_n\',\'novel_gene_preserved\',\'novel_gene_supported\',\'novel_type_preserved\',\'novel_type_supported\',\'novel_aa_pair_preserved\',\'novel_aa_pair_supported\',\'novel_aa_components_preserved\',\'novel_aa_components_supported\',\'novel_no_supported_upper\',\'unseen_exact_known_site\',\'unseen_site_supported_type\']:\n                d[key]=0\n            for key in s[\'exact\']-self.seen[\'exact\']:\n                _,gene,token=key.split(\'|\',2);p=parse_token(token);d[\'novel_tokens\']+=1\n                gk=f\'gene|{gene}|nonWT\';tk=f\'type|{gene}|{p["type"]}\'\n                d[\'novel_gene_preserved\']+=int(gk in self.index);d[\'novel_gene_supported\']+=int(self.observed_support[gk]>0)\n                if p[\'type\']!=\'unparsed\':\n                    d[\'novel_parsed_type_n\']+=1;d[\'novel_type_preserved\']+=int(tk in self.index);d[\'novel_type_supported\']+=int(self.observed_support[tk]>0)\n                upper=[gk,tk,\'global|type_fraction|\'+p[\'type\']]\n                if p[\'change\'] and p[\'aa_from\'] in AA and p[\'aa_to\'] in AA:\n                    ak=\'aa|\'+p[\'change\'];fk=\'aa_from|\'+p[\'aa_from\'];bk=\'aa_to|\'+p[\'aa_to\'];d[\'novel_aa_n\']+=1\n                    d[\'novel_aa_pair_preserved\']+=int(ak in self.index);d[\'novel_aa_pair_supported\']+=int(self.observed_support[ak]>0)\n                    d[\'novel_aa_components_preserved\']+=int(fk in self.index and bk in self.index)\n                    d[\'novel_aa_components_supported\']+=int(self.observed_support[fk]>0 and self.observed_support[bk]>0)\n                    upper.extend([ak,fk,bk])\n                d[\'novel_no_supported_upper\']+=int(not any(self.observed_support[k]>0 for k in upper))\n                if p[\'site\']:\n                    site=f\'site|{gene}|{p["site"]}\'\n                    d[\'unseen_exact_known_site\']+=int(site in self.seen[\'site\'])\n                    d[\'unseen_site_supported_type\']+=int(site not in self.seen[\'site\'] and self.observed_support[gk]>0 and self.observed_support[tk]>0)\n            rows.append(d)\n        return pd.DataFrame(rows)\n')
Path('/kaggle/working/test_cbj_anchor_features.py').write_text("import unittest\nimport pandas as pd\nimport numpy as np\nfrom cbj_anchor_features import parse_token, build_samples, AnchorEncoder\n\n\nclass AnchorContractTests(unittest.TestCase):\n    def test_parser_distinguishes_synonymous_and_loss_types(self):\n        self.assertEqual(parse_token('D623D')['type'], 'synonymous')\n        self.assertEqual(parse_token('p.R132H')['site'], '132')\n        self.assertEqual(parse_token('R132H')['change'], 'R>H')\n        self.assertEqual(parse_token('E237*')['type'], 'stop_gained')\n        self.assertEqual(parse_token('L1854fs')['type'], 'frameshift')\n        self.assertEqual(parse_token('K857delinsNDLQ')['type'], 'delins')\n        self.assertEqual(parse_token('GD693fs')['site'], '')\n        self.assertEqual(parse_token('1529_1530AL>AL')['type'], 'complex')\n        self.assertEqual(parse_token('unexpected')['type'], 'unparsed')\n\n    def test_permutation_repetition_and_missing(self):\n        df=pd.DataFrame({'ID':['a','b','c'],'G':['R1H A2A R1H','A2A R1H',''],'H':['WT','WT','WT']})\n        samples,ann,missing=build_samples(df,['G','H'],{})\n        self.assertEqual(samples[0]['canonical_hash'],samples[1]['canonical_hash'])\n        self.assertEqual(len(ann),5)\n        self.assertEqual(samples[0]['features']['gene|G|nonWT'],1)\n        self.assertEqual(samples[2]['features']['gene|G|missing'],1)\n        self.assertNotIn('gene|G|nonWT',samples[2]['features'])\n        self.assertEqual(samples[0]['repeat_tokens'],1)\n        enc=AnchorEncoder(['G','H'],{},min_support=1).fit(samples)\n        self.assertEqual((enc.transform(samples[:1])!=enc.transform(samples[1:2])).nnz,0)\n\n    def test_fold_vocab_unseen_fallback_and_serialization(self):\n        df=pd.DataFrame({'ID':['a','b','c'],'G':['R1H','R1H','R999C'],'H':['WT']*3})\n        samples,_,_=build_samples(df,['G','H'],{})\n        enc=AnchorEncoder(['G','H'],{},min_support=2).fit(samples[:2])\n        self.assertNotIn('exact|G|R999C',enc.names)\n        before=enc.to_dict();x=enc.transform(samples[2:])\n        self.assertEqual(before,enc.to_dict())\n        self.assertEqual(x[0,enc.index['gene|G|nonWT']],1)\n        self.assertEqual(x[0,enc.index['type|G|missense']],1)\n        self.assertGreater(x[0,enc.index['aa|R>C']],0)\n        self.assertEqual((AnchorEncoder.from_dict(before).transform(samples[2:])!=x).nnz,0)\n        novel=enc.novelty(samples[2:]).iloc[0]\n        self.assertEqual(novel.novel_aa_pair_preserved,1)\n        self.assertEqual(novel.novel_aa_pair_supported,0)\n        self.assertEqual(novel.novel_aa_components_supported,0)\n\n    def test_relationship_notation_and_empty_oov_denominator(self):\n        df=pd.DataFrame({'ID':['a','b'],'G':['[R1H;A2C]','WT']})\n        s,a,_=build_samples(df,['G'],{})\n        self.assertEqual(len(a),1)\n        self.assertEqual(a.iloc[0]['type'],'unparsed')\n        self.assertEqual(a.iloc[0]['raw'],'[R1H;A2C]')\n        enc=AnchorEncoder(['G'],{},1).fit(s)\n        self.assertTrue(np.isnan(enc.novelty(s[1:]).iloc[0].unseen_exact_fraction))\n\n    def test_rare_support_counts_profiles_not_duplicate_rows(self):\n        df=pd.DataFrame({'ID':['a','b','c'],'G':['R1H','R1H','R2C']})\n        s,_,_=build_samples(df,['G'],{})\n        enc=AnchorEncoder(['G'],{},2).fit(s)\n        self.assertIn('exact|G|R1H',enc.seen['exact'])\n        self.assertNotIn('exact|G|R1H',enc.index)\n        self.assertEqual(enc.observed_support['gene|G|nonWT'],2)\n\n    def test_missing_column_and_column_order(self):\n        df=pd.DataFrame({'ID':['a'],'G':['R1H'],'H':['WT']})\n        a,_,_=build_samples(df,['G','H'],{})\n        b,_,_=build_samples(df[['H','ID','G']],['G','H'],{})\n        self.assertEqual(a[0]['features'],b[0]['features'])\n        c,_,_=build_samples(df.drop(columns='H'),['G','H'],{})\n        self.assertEqual(c[0]['features']['gene|H|missing'],1)\n        with self.assertRaises(ValueError):build_samples(df.assign(EXTRA='R1H'),['G','H'],{})\n\n    def test_pathway_unknown_is_not_zero_wt(self):\n        df=pd.DataFrame({'ID':['a','b'],'G':['','R1H'],'H':['WT','']})\n        s,_,_=build_samples(df,['G','H'],{'stage':['G','H']})\n        self.assertEqual(s[0]['features']['path|stage|unknown'],1)\n        self.assertNotIn('path|stage|any',s[0]['features'])\n        self.assertEqual(s[1]['features']['path|stage|any'],1)\n        self.assertNotIn('path|stage|unknown',s[1]['features'])\n        self.assertGreater(s[1]['features']['path|stage|missing_fraction'],0)\n\n\nif __name__=='__main__':unittest.main()\n")
Path('/kaggle/working/cbj_anchor_train.py').write_text('"""Prepared training entrypoint; never invoked by the preparation notebook.\n\nRun only after training is authorized: python cbj_anchor_train.py --train --prepared DIR\n"""\nimport argparse\nimport json\nimport hashlib\nfrom pathlib import Path\nimport warnings\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import load_npz\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.metrics import f1_score,accuracy_score,balanced_accuracy_score,log_loss,classification_report\nimport joblib\n\n\ndef main():\n    ap=argparse.ArgumentParser();ap.add_argument(\'--train\',action=\'store_true\');ap.add_argument(\'--prepared\',type=Path,required=True)\n    args=ap.parse_args()\n    if not args.train:raise SystemExit(\'Training is disabled. Explicit --train is required.\')\n    if not Path(\'/kaggle/working\').exists():raise SystemExit(\'Training is restricted to the authorized Kaggle environment.\')\n    root=args.prepared;config=json.loads((root/\'anchor_config.json\').read_text())\n    receipt=json.loads((root/\'preflight.json\').read_text())\n    if not receipt[\'ready_for_training\']:raise SystemExit(\'Preparation gate failed.\')\n    for name,expected in receipt[\'file_sha256\'].items():\n        if hashlib.sha256((root/name).read_bytes()).hexdigest()!=expected:raise SystemExit(\'Artifact changed: \'+name)\n    folds=pd.read_csv(root/\'fold_assignments.csv\');classes=config[\'class_order\'];y=folds.SUBCLASS.map({c:i for i,c in enumerate(classes)}).to_numpy()\n    probs=np.full((len(y),len(classes)),np.nan);reports=[]\n    output=root/\'trained_anchor\';output.mkdir(exist_ok=True)\n    for fold in range(config[\'n_folds\']):\n        tr=np.flatnonzero(folds.fold.to_numpy()!=fold);va=np.flatnonzero(folds.fold.to_numpy()==fold)\n        xtr=load_npz(root/f\'fold_{fold}_train.npz\');xva=load_npz(root/f\'fold_{fold}_valid.npz\')\n        assert xtr.shape[0]==len(tr) and xva.shape[0]==len(va)\n        model=LogisticRegression(**config[\'model_params\'])\n        with warnings.catch_warnings(record=True) as seen:\n            warnings.simplefilter(\'always\',ConvergenceWarning);model.fit(xtr,y[tr])\n        converged=not any(issubclass(w.category,ConvergenceWarning) for w in seen)\n        assert list(model.classes_)==list(range(len(classes)))\n        p=model.predict_proba(xva);probs[va]=p;pred=p.argmax(axis=1)\n        reports.append({\'fold\':fold,\'converged\':converged,\'max_iterations_used\':int(model.n_iter_.max()),\n                        \'macro_f1\':float(f1_score(y[va],pred,labels=np.arange(len(classes)),average=\'macro\',zero_division=0))})\n        joblib.dump(model,output/f\'fold_{fold}_model.joblib\',compress=3)\n    assert np.isfinite(probs).all() and np.allclose(probs.sum(axis=1),1,atol=1e-6)\n    result=folds[[\'ID\',\'SUBCLASS\',\'fold\',\'canonical_hash\']].copy()\n    for i,cl in enumerate(classes):result[\'p_\'+cl]=probs[:,i]\n    novelty=pd.read_csv(root/\'oof_novelty_by_patient.csv\').set_index(\'ID\').loc[folds.ID]\n    for col in novelty.columns:\n        if col not in result:result[col]=novelty[col].values\n    patients=pd.read_csv(root/\'patient_summary.csv\').set_index(\'ID\').loc[folds.ID]\n    result[\'missing_gene_count\']=patients.missing_gene_count.values\n    result[\'representation_version\']=config[\'representation_version\']\n    result.to_csv(output/\'oof_probabilities.csv\',index=False)\n    pred=probs.argmax(axis=1)\n    summary={\'scope\':\'grouped internal OOF; not external validation\',\'fold_reports\':reports,\n             \'macro_f1\':float(f1_score(y,pred,labels=np.arange(len(classes)),average=\'macro\',zero_division=0)),\n             \'accuracy\':float(accuracy_score(y,pred)),\'balanced_accuracy\':float(balanced_accuracy_score(y,pred)),\n             \'log_loss\':float(log_loss(y,probs,labels=np.arange(len(classes))))}\n    pd.DataFrame(classification_report(y,pred,labels=np.arange(len(classes)),target_names=classes,output_dict=True,zero_division=0)).T.to_csv(output/\'per_class.csv\')\n    bins={\'seen_exact\':novelty.exact_unseen.eq(0)&novelty.exact_n.gt(0),\'no_variant\':novelty.exact_n.eq(0),\'some_unseen_exact\':novelty.exact_unseen.gt(0),\n          \'mostly_unseen_exact\':novelty.unseen_exact_fraction.ge(.5),\'unseen_site\':novelty.site_unseen.gt(0),\'no_supported_upper\':novelty.novel_no_supported_upper.gt(0)}\n    summary[\'novelty_subgroups\']={}\n    for name,mask in bins.items():\n        ix=np.flatnonzero(mask.to_numpy())\n        summary[\'novelty_subgroups\'][name]={\'n\':len(ix),\'macro_f1_all_26_classes\':float(f1_score(y[ix],pred[ix],labels=np.arange(len(classes)),average=\'macro\',zero_division=0)) if len(ix) else None}\n    class_subgroups=[]\n    burden_bins=pd.cut(patients.mutated_gene_count,[-1,5,10,20,50,100,np.inf]).astype(str).values\n    for name,mask in bins.items():\n        for cl in range(len(classes)):\n            ix=np.flatnonzero(mask.to_numpy()&(y==cl))\n            if len(ix):class_subgroups.append({\'novelty_group\':name,\'SUBCLASS\':classes[cl],\'n\':len(ix),\'recall\':float(np.mean(pred[ix]==y[ix])),\'mean_true_class_log_loss\':float(-np.log(np.clip(probs[ix,cl],1e-15,1)).mean())})\n    pd.DataFrame(class_subgroups).to_csv(output/\'novelty_class_metrics.csv\',index=False)\n    result.assign(burden_bin=burden_bins).to_csv(output/\'oof_probabilities.csv\',index=False)\n    summary[\'fold_macro_f1_mean\']=float(np.mean([r[\'macro_f1\'] for r in reports]))\n    summary[\'fold_macro_f1_std\']=float(np.std([r[\'macro_f1\'] for r in reports]))\n    summary[\'all_folds_converged\']=all(r[\'converged\'] for r in reports)\n    (output/\'metrics.json\').write_text(json.dumps(summary,indent=2))\n    if not summary[\'all_folds_converged\']:raise SystemExit(\'Convergence gate failed; metrics are diagnostic, not an accepted anchor.\')\n    cold=pd.read_csv(root/\'cold_site_split.csv\');cy=cold.SUBCLASS.map({c:i for i,c in enumerate(classes)}).to_numpy()\n    tr=np.flatnonzero(cold.split.eq(\'train\'));va=np.flatnonzero(cold.split.eq(\'cold_validation\'))\n    cm=LogisticRegression(**config[\'model_params\'])\n    with warnings.catch_warnings(record=True) as seen:\n        warnings.simplefilter(\'always\',ConvergenceWarning);cm.fit(load_npz(root/\'cold_train.npz\'),cy[tr])\n    cp=cm.predict_proba(load_npz(root/\'cold_valid.npz\'))\n    cr=cold.iloc[va].copy()\n    for i,cl in enumerate(classes):cr[\'p_\'+cl]=cp[:,i]\n    cr.to_csv(output/\'cold_site_probabilities.csv\',index=False)\n    present=sorted(set(cy[va]))\n    cold_metrics={\'scope\':\'restricted cold-site scenario; different class composition from OOF\',\'n\':len(va),\'present_classes\':[classes[i] for i in present],\n                  \'macro_f1_present_classes\':float(f1_score(cy[va],cp.argmax(axis=1),labels=present,average=\'macro\',zero_division=0)),\n                  \'log_loss\':float(log_loss(cy[va],cp,labels=np.arange(len(classes)))),\'converged\':not any(issubclass(w.category,ConvergenceWarning) for w in seen)}\n    (output/\'cold_site_metrics.json\').write_text(json.dumps(cold_metrics,indent=2))\n    # No full refit/test prediction here: inspect OOF and authorize those separately.\n    print(json.dumps(summary,indent=2))\n\n\nif __name__==\'__main__\':main()\n')
sys.path.insert(0,'/kaggle/working')
from cbj_anchor_features import build_samples,AnchorEncoder,TYPES,AA,VERSION,parse_token
from test_cbj_anchor_features import AnchorContractTests
suite=unittest.defaultTestLoader.loadTestsFromTestCase(AnchorContractTests)
test_result=unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()
import pandas as pd,numpy as np
from scipy.sparse import save_npz
from sklearn.model_selection import StratifiedGroupKFold
import sklearn,matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager,colors
from IPython.display import display,Image,Markdown
matches=list(Path('/kaggle/input').rglob('cbj_gene_pathway.csv'));assert len(matches)==1
DATA=matches[0].parent
font_manager.fontManager.addfont(str(DATA/'NanumGothic-Regular.ttf'))
plt.rcParams.update({'font.family':'NanumGothic','font.size':11,'axes.unicode_minus':False,'figure.dpi':120,'savefig.dpi':160})
frames={name:pd.read_csv(DATA/name,dtype=str,keep_default_na=False) for name in ['train.csv','test.csv','sample_submission.csv']}
train,test,submission=[frames[k] for k in frames];genes=[g for g in train if g not in ['ID','SUBCLASS']]
assert genes==[g for g in test if g!='ID'] and test.ID.tolist()==submission.ID.tolist()
mapping=pd.read_csv(matches[0],keep_default_na=False)
stages=list(dict.fromkeys(mapping.stage))
stage_genes={s:mapping.loc[(mapping.stage==s)&mapping.gene.isin(genes),'gene'].drop_duplicates().tolist() for s in stages}
SCOPE=f'전체 train {len(train):,}명 / {len(genes):,}개 유전자 · 표기 기반 분석'
print('Full source rows:',{k:len(v) for k,v in frames.items()},flush=True)
t0=time.time();samples,ann,miss=build_samples(train,genes,stage_genes)
test_samples,test_ann,test_miss=build_samples(test,genes,stage_genes)
ann=ann.merge(train[['ID','SUBCLASS']],on='ID',validate='many_to_one')
u=ann.drop_duplicates(['ID','gene','normalized_token']).copy()
patients=pd.DataFrame([{k:s[k] for k in ['ID','canonical_hash','raw_hash','mutated_gene_count','missing_gene_count','unique_token_count','token_count','repeat_tokens']} for s in samples]).merge(train[['ID','SUBCLASS']],on='ID',validate='one_to_one')
print('Parsing complete',len(ann),'raw fragments;',len(u),'unique sample-gene-token rows;',round(time.time()-t0,1),'s',flush=True)
def save(fig,name):
    fig.savefig(OUT/(name+'.png'),bbox_inches='tight',facecolor='white');plt.close(fig)
    display(Image(filename=str(OUT/(name+'.png'))))
def heat(ax,values,xlabels,ylabels,annot=None,vmax=None):
    a=np.array(values,dtype=float);cm=plt.get_cmap('Blues').copy();cm.set_bad('#ccc')
    im=ax.imshow(np.ma.masked_invalid(a),aspect='auto',cmap=cm,vmin=0,vmax=vmax)
    ax.set_xticks(range(len(xlabels)),xlabels,rotation=45,ha='right');ax.set_yticks(range(len(ylabels)),ylabels)
    if annot is not None:
        for i,row in enumerate(annot):
            for j,t in enumerate(row):ax.text(j,i,t,ha='center',va='center',fontsize=9,color='white' if np.isfinite(a[i,j]) and a[i,j]>(vmax or np.nanmax(a))*.65 else '#222')
    return im

## 1. 전체 유전자·변이 유형: 정보의 양과 구성
환자당 비WT 유전자 수와 유형별 고유 환자-유전자 수를 봅니다. 한 유전자에 여러 유형이 있으면 유형 합계는 겹칠 수 있습니다. 높은 변이 수를 이상치로 자동 삭제하지 않습니다.

In [ ]:
classes=sorted(train.SUBCLASS.unique());counts=train.SUBCLASS.value_counts().reindex(classes)
type_cells=u.drop_duplicates(['ID','gene','type'])
global_types=type_cells.groupby('type').size().reindex(TYPES,fill_value=0)
fig,(ax,bx)=plt.subplots(1,2,figsize=(16,9),gridspec_kw={'width_ratios':[1.6,1]})
vals=[patients.loc[patients.SUBCLASS==c,'mutated_gene_count'].values for c in classes]
ax.boxplot(vals,vert=False,showfliers=False);ax.set_yticks(range(1,len(classes)+1),[f'{c} n={counts[c]}' for c in classes]);ax.invert_yaxis();ax.set_xscale('symlog',linthresh=1);ax.set_xlim(left=0);ax.set_xticks([0,1,10,100,1000],[str(v) for v in [0,1,10,100,1000]])
ax.set_xlabel('환자당 비WT 유전자 수 (symlog; 수염 밖 점만 생략, 계산에는 전원 포함)');ax.set_title('SUBCLASS별 전체 변이 유전자 수')
bx.barh(TYPES,global_types,color='#467eae');bx.invert_yaxis();bx.set_xscale('symlog',linthresh=1);bx.set_xlim(left=0);bx.set_xticks([0,1,10,100,1000,10000,100000],['0','1','10','100','1000','10000','100000']);bx.tick_params(axis='x',labelrotation=35);bx.set_xlabel('고유 환자-유전자 수 (유형 간 중복 가능)');bx.set_title('전체 유전자 변이 유형')
for i,v in enumerate(global_types):bx.text(v*1.05+1,i,f'{v:,}',va='center',fontsize=10)
fig.suptitle('앵커 EDA 1 | 전체 유전자와 변이 유형',fontsize=18,y=1.01)
fig.text(.02,-.02,SCOPE+' · 작은 경로 목록으로 유전자를 선별하지 않음',fontsize=11)
fig.tight_layout();save(fig,'01_genome_type_overview')
type_cells.groupby(['SUBCLASS','type']).size().rename('patient_gene_cells').reset_index().to_csv(OUT/'class_type_counts.csv',index=False)
patients.groupby('SUBCLASS').mutated_gene_count.agg(['count','median','mean','min','max']).to_csv(OUT/'burden_by_class.csv')
display(Markdown('무엇을 보는가: 변이 유전자 수와 유형 구성. 현재 보이는 값은 그림과 표의 전체 집계입니다. 다음 확인: 총 변이 수를 고려한 뒤에도 개별 유형·위치가 정보를 더하는지.'))

## 2. 위치와 아미노산을 함께 보기
왼쪽은 전체 유전자에서 파싱된 단순 아미노산 치환의 분포(동의 포함), 오른쪽은 가장 많은 환자에서 비WT가 관측된 4개 유전자의 위치 분포입니다. 빈도 기준으로 선택했으며 예측 중요도 순위가 아닙니다. 위치는 각 유전자 내부의 표기상 단백질 좌표입니다. 전사체·참조서열 일치는 미검증이며 불확실한 위치는 그리지 않습니다.

In [ ]:
simple=u[u.type.isin(['synonymous','missense']) & u.aa_from.isin(list(AA)) & u.aa_to.isin(list(AA))]
aa_table=pd.crosstab(simple.aa_from,simple.aa_to).reindex(index=list(AA),columns=list(AA),fill_value=0).fillna(0).astype(int)
fig,ax=plt.subplots(figsize=(11,9));im=ax.imshow(aa_table,cmap='Blues',norm=colors.LogNorm(vmin=1,vmax=max(int(aa_table.values.max()),2)))
ax.set_xticks(range(20),list(AA));ax.set_yticks(range(20),list(AA));ax.set_xlabel('변화 후 아미노산');ax.set_ylabel('변화 전 아미노산')
cb=fig.colorbar(im,ax=ax,label='고유 환자-유전자-표기 수 (로그 색상; 빈 셀=0)');ticks=[v for v in [1,10,100,1000,10000,100000] if v<=aa_table.values.max()];cb.set_ticks(ticks);cb.set_ticklabels([str(v) for v in ticks])
ax.set_title('앵커 EDA 2a | 전체 유전자의 아미노산 치환',pad=20)
fig.text(.02,-.01,SCOPE+f' · 단순 치환 {len(simple):,}개 표기 · 대각선은 동의 표기; 위험도 점수 아님',fontsize=10)
fig.tight_layout();save(fig,'02a_amino_acid_changes');aa_table.to_csv(OUT/'aa_change_counts.csv')
topgenes=u.drop_duplicates(['ID','gene']).gene.value_counts().head(4).index.tolist()
fig,axs=plt.subplots(4,1,figsize=(16,10))
pos_rows=[]
for ax,g in zip(axs,topgenes):
    z=u[(u.gene==g)&u.parse_status.isin(['simple_protein_notation','frameshift_start_only'])].copy()
    for typ,col,marker in [('synonymous','#bf8b3a','o'),('missense','#467eae','o'),('stop_gained','#222222','x'),('frameshift','#777777','^')]:
        q=z[z.type==typ].groupby('position_start').ID.nunique()
        ax.scatter(q.index,q.values,label=typ,c=col,s=18,marker=marker)
        for p,n in q.items():pos_rows.append({'gene':g,'type':typ,'protein_position':p,'patients':n})
    ax.set_title(f'{g} | 비WT 환자 {u.loc[u.gene==g,"ID"].nunique():,}명',loc='left');ax.set_ylabel('환자 수');ax.set_xlabel('해당 유전자의 단백질 표기 위치');ax.legend(ncol=4,fontsize=9)
fig.suptitle('앵커 EDA 2b | 전체 유전자 빈도 상위 4개의 위치·유형',fontsize=18,y=1.01)
fig.text(.02,-.02,SCOPE+' · 같은 위치·유형의 환자 중복 제거; 유전자 간 위치 크기를 비교하지 않음',fontsize=11)
fig.tight_layout();save(fig,'02b_gene_positions');pd.DataFrame(pos_rows).to_csv(OUT/'gene_position_counts.csv',index=False)
display(u[u.gene.isin(topgenes)].groupby(['gene','normalized_token','type']).ID.nunique().sort_values(ascending=False).head(25).rename('patients').reset_index())

## 3. 입력 표현의 중복과 라벨 혼동
공백·표기 순서·반복 토큰·선택적 `p.` 접두사를 정규화한 별도 canonical profile을 만듭니다. 원본은 보존합니다. 동일 profile은 같은 fold로 묶고 라벨을 삭제·병합하지 않습니다. 같은 profile의 다른 라벨은 환자 중복이나 오라벨의 증거로 확정하지 않습니다.

In [ ]:
canon=patients.groupby('canonical_hash').agg(n=('ID','size'),classes=('SUBCLASS',lambda x:sorted(set(x))))
conflict=canon[canon.classes.map(len)>1];raw_groups=patients.groupby('raw_hash').SUBCLASS.nunique()
pair_counts={}
for labels in conflict.classes:
    for a,b in itertools.combinations(labels,2):pair_counts[(a,b)]=pair_counts.get((a,b),0)+1
pair_table=pd.DataFrame([{'class_a':a,'class_b':b,'canonical_groups':n} for (a,b),n in pair_counts.items()]).sort_values('canonical_groups',ascending=False)
pair_table.to_csv(OUT/'conflicting_profile_class_pairs.csv',index=False)
fig,ax=plt.subplots(figsize=(12,6));q=pair_table.head(12)
ax.barh(q.class_a+' / '+q.class_b,q.canonical_groups,color='#467eae');ax.invert_yaxis();ax.set_xlabel('두 라벨이 함께 있는 canonical profile 그룹 수')
for i,v in enumerate(q.canonical_groups):ax.text(v+.3,i,str(v),va='center')
ax.set_title('앵커 EDA 3 | 동일 입력의 상충 라벨쌍 상위 12개',pad=16)
fig.text(.02,-.02,SCOPE+' · 단위=profile 그룹, 환자 수 아님 · 라벨 정의 확인이 우선',fontsize=11)
fig.tight_layout();save(fig,'03_profile_label_conflicts')
display(pd.DataFrame([{'raw_duplicate_extra_rows':len(train)-patients.raw_hash.nunique(),'canonical_duplicate_extra_rows':len(train)-patients.canonical_hash.nunique(),'raw_conflicting_groups':int((raw_groups>1).sum()),'canonical_conflicting_groups':len(conflict),'repeated_notation_fragments':int(patients.repeat_tokens.sum())}]))

## 4. 같은 주요 변이 안에서 나머지 유전자·유형 보기
IDH1 R132H 양성 LGG·GBMLGG에 조건을 맞추고, 그 안에서 자주 관측된 다른 유전자 8개를 봅니다. 이는 학습 피처를 선택하는 규칙이 아니라 상세 EDA입니다. 단계만 보지 않고 위치·유형·전체 유전자 맥락을 연결합니다. 총 변이 수 구간별 분모도 CSV에 보존합니다.

In [ ]:
ids=set(u.loc[(u.gene=='IDH1')&(u.normalized_token=='R132H'),'ID'])
sub=patients[patients.ID.isin(ids)&patients.SUBCLASS.isin(['LGG','GBMLGG'])].copy()
focus=u[u.ID.isin(sub.ID)&u.gene.ne('IDH1')]
focusgenes=focus.drop_duplicates(['ID','gene']).gene.value_counts().head(8).index.tolist()
rows=[];labs=[];arr=[];txt=[]
for gene in focusgenes:
    for typ in ['synonymous','missense','stop_gained','frameshift']:
        rr=[];tt=[]
        for cl in ['LGG','GBMLGG']:
            target=set(sub.loc[sub.SUBCLASS==cl,'ID']);num=focus.loc[(focus.gene==gene)&(focus.type==typ)&focus.ID.isin(target),'ID'].nunique();den=len(target)
            rr.append(num/den if den else np.nan);tt.append(f'{num}/{den}');rows.append({'gene':gene,'type':typ,'SUBCLASS':cl,'positive':num,'known':den})
        arr.append(rr);txt.append(tt);labs.append(gene+' | '+typ)
fig,ax=plt.subplots(figsize=(10,max(6,len(labs)*.26)))
im=heat(ax,arr,['LGG','GBMLGG'],labs,txt,vmax=1);fig.colorbar(im,ax=ax,label='IDH1 R132H 양성군 내 환자 비율')
ax.set_title('앵커 EDA 4 | 같은 IDH1 표기 안의 다른 유전자·유형',pad=18)
fig.text(.02,-.025,SCOPE+f' · 조건부 {len(sub)}명 · 관측 연관성; 상충 profile 포함',fontsize=10)
fig.tight_layout();save(fig,'04_conditional_gene_types')
sub['burden_bin']=pd.cut(sub.mutated_gene_count,[-1,5,10,20,50,100,np.inf]).astype(str)
sub.groupby(['SUBCLASS','burden_bin']).size().rename('patients').reset_index().to_csv(OUT/'conditional_burden_denominators.csv',index=False)
pd.DataFrame(rows).to_csv(OUT/'conditional_gene_type_rates.csv',index=False)

## 5. 모든 관점을 포함한 공통 입력과 신규 표기 감사
고정 블록: 모든 유전자의 비WT·결측 및 유형, 전역 아미노산 치환 비율, 전체 변이 수 등 소수 요약, 관측 가능한 기능 단계의 작은 보조 블록.

세부 블록: `gene+site`, `gene+AA change`, `gene+exact token`. **학습 fold 내 서로 다른 canonical profile 3개 이상**에서 본 세부 표현만 vocabulary에 포함합니다. 검증·test에서 처음 본 세부 표현은 vocabulary를 바꾸지 않고 상위 유전자·유형·아미노산 표현으로 연결합니다. 일반적인 치환 규칙은 사전 고정이며 라벨을 쓰지 않습니다.

모든 값은 유무 또는 명시된 분모로 정규화된 0~1 값입니다. 범주 ID나 단백질 위치의 크기를 연속 피처로 사용하지 않습니다. 아래 수치는 예측 성능이 아니라 처음 보는 세부 표기의 비율과 남는 표현입니다.

In [ ]:
class_order=sorted(classes);y=train.SUBCLASS.map({c:i for i,c in enumerate(class_order)}).to_numpy()
groups=patients.canonical_hash.to_numpy();foldids=np.full(len(train),-1)
splitter=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
checks=[];novels=[];fold_summary=[];feature_counts=[];collision_audit=[]
for fold,(tr,va) in enumerate(splitter.split(np.zeros(len(train)),y,groups)):
    assert set(groups[tr]).isdisjoint(groups[va]);assert set(y[tr])==set(range(len(class_order)))
    assert set(y[va])==set(range(len(class_order)))
    foldids[va]=fold
    training=[samples[i] for i in tr];validation=[samples[i] for i in va]
    enc=AnchorEncoder(genes,stage_genes,min_support=3).fit(training)
    xtr=enc.transform(training);xva=enc.transform(validation)
    assert set(enc.fit_ids)==set(train.ID.iloc[tr]);assert not set(enc.fit_ids)&set(train.ID.iloc[va])
    assert xtr.shape[1]==xva.shape[1] and xtr.shape[0]==len(tr)
    assert xtr.data.min(initial=0)>=0 and xtr.data.max(initial=0)<=1
    assert np.isfinite(xva.data).all()
    def matrix_hashes(x):
        x=x.copy();x.sort_indices()
        return [hashlib.sha256(x.indices[x.indptr[i]:x.indptr[i+1]].tobytes()+x.data[x.indptr[i]:x.indptr[i+1]].tobytes()).hexdigest() for i in range(x.shape[0])]
    profiles=pd.DataFrame({'row_hash':matrix_hashes(xtr)+matrix_hashes(xva),'profile':list(groups[tr])+list(groups[va]),'label':list(train.SUBCLASS.iloc[tr])+list(train.SUBCLASS.iloc[va]),'split':['train']*len(tr)+['valid']*len(va)})
    collisions=profiles.groupby('row_hash').agg(profiles=('profile','nunique'),labels=('label','nunique'),splits=('split','nunique'))
    cross=collisions[(collisions.profiles>1)&(collisions.splits>1)]
    collision_audit.append({'fold':fold,'distinct_profiles_collapsed_groups':int((collisions.profiles>1).sum()),'cross_split_collision_groups':len(cross),'cross_split_affected_valid_n':int(((profiles.split=='valid')&profiles.row_hash.isin(cross.index)).sum()),'conflicting_label_feature_groups':int((collisions.labels>1).sum())})
    vocab_bytes=json.dumps(enc.to_dict(),sort_keys=True).encode()
    frozen_hash=hashlib.sha256(vocab_bytes).hexdigest()
    novel=enc.novelty(validation);novel['fold']=fold;novel['SUBCLASS']=train.SUBCLASS.iloc[va].values
    assert novel.loc[novel.mutated_gene_count>0,'coarse_nonzero'].gt(0).all()
    # Transform does not mutate training vocabulary, including for novel test patterns.
    assert hashlib.sha256(json.dumps(enc.to_dict(),sort_keys=True).encode()).hexdigest()==frozen_hash
    with gzip.open(OUT/f'fold_{fold}_encoder.json.gz','wt') as f:json.dump(enc.to_dict(),f)
    restored=AnchorEncoder.from_dict(enc.to_dict())
    assert (restored.transform(validation[:5])!=xva[:5]).nnz==0
    save_npz(OUT/f'fold_{fold}_train.npz',xtr);save_npz(OUT/f'fold_{fold}_valid.npz',xva)
    novels.append(novel)
    summary={'fold':fold,'train_n':len(tr),'valid_n':len(va),'features':len(enc.names),'train_nonzero':xtr.nnz,'valid_nonzero':xva.nnz,'train_profiles':enc.fit_group_count}
    for block in ['exact','site','gene_change']:
        den=int(novel[block+'_n'].sum());summary[block+'_den']=den
        for suffix in ['unseen','filtered_seen','retained']:summary[block+'_'+suffix+'_fraction']=float(novel[block+'_'+suffix].sum()/max(den,1))
    fold_summary.append(summary)
    feature_counts.append({'fold':fold,**pd.Series([name.split('|')[0] for name in enc.names]).value_counts().to_dict()})
    checks.append({'fold':fold,'group_overlap':0,'all_classes_present':True,'finite_matrix':True,'vocabulary_frozen':True,'serialization_equal':True})
    print('Prepared fold',fold,len(tr),len(va),len(enc.names),'features',flush=True)
assert (foldids>=0).all()
assignments=train[['ID','SUBCLASS']].copy();assignments['fold']=foldids;assignments['canonical_hash']=groups;assignments['raw_hash']=patients.raw_hash
assignments.to_csv(OUT/'fold_assignments.csv',index=False)
assert assignments.groupby('canonical_hash').fold.nunique().eq(1).all()
assignments.groupby(['fold','SUBCLASS']).agg(patients=('ID','size'),canonical_groups=('canonical_hash','nunique')).reset_index().to_csv(OUT/'fold_class_support.csv',index=False)
pd.DataFrame(collision_audit).to_csv(OUT/'feature_row_collisions.csv',index=False)
novel_all=pd.concat(novels,ignore_index=True);assert novel_all.ID.nunique()==len(train)
novel_all.to_csv(OUT/'oof_novelty_by_patient.csv',index=False)
pd.DataFrame(fold_summary).to_csv(OUT/'fold_novelty_summary.csv',index=False);pd.DataFrame(feature_counts).to_csv(OUT/'feature_block_sizes.csv',index=False)
# Full-train encoder is ONLY for deployment preparation and label-free test novelty audit.
full_enc=AnchorEncoder(genes,stage_genes,min_support=3).fit(samples)
full_vocab_hash=hashlib.sha256(json.dumps(full_enc.to_dict(),sort_keys=True).encode()).hexdigest()
test_novel=full_enc.novelty(test_samples);xtest=full_enc.transform(test_samples)
assert full_vocab_hash==hashlib.sha256(json.dumps(full_enc.to_dict(),sort_keys=True).encode()).hexdigest()
test_novel.to_csv(OUT/'test_novelty_audit.csv',index=False)
with gzip.open(OUT/'full_encoder.json.gz','wt') as f:json.dump(full_enc.to_dict(),f)
save_npz(OUT/'all_train.npz',full_enc.transform(samples));save_npz(OUT/'unlabeled_test.npz',xtest)
pd.DataFrame({'ID':test.ID}).to_csv(OUT/'test_row_order.csv',index=False)
table=pd.DataFrame(fold_summary)
fig,(ax,bx)=plt.subplots(1,2,figsize=(15,5.5))
labels=['exact','site','gene_change'];values=table[[k+'_unseen_fraction' for k in labels]].values
text=[[f'{v:.1%}' for v in row] for row in values]
im=heat(ax,values,labels,[f'fold {i} (n={int(table.loc[i,"valid_n"])})' for i in range(5)],text,vmax=1)
ax.set_title('学習 fold에서 전혀 보지 못한 세부 표기 비율'.replace('学習','학습'),pad=16)
test_vals=[test_novel[k+'_unseen'].sum()/max(test_novel[k+'_n'].sum(),1) for k in labels]
bx.bar(labels,test_vals,color='#467eae');bx.set_ylim(0,1);bx.set_ylabel('고유 환자 내 세부 키 중 미지 키 비율');bx.set_title(f'test {len(test):,}명: 전체 train 기준 감사',pad=16)
for i,v in enumerate(test_vals):bx.text(i,v+.02,f'{v:.1%}',ha='center')
fig.suptitle('앵커 EDA 5 | 처음 보는 변이에 남는 정보',fontsize=18,y=1.02)
fig.text(.02,-.03,SCOPE+' · 분모=각 환자의 고유 세부 키 합계 · 미지 세부 표기에도 고정 gene/type/AA 표현 유지; 정확도 향상은 미검증',fontsize=10)
fig.tight_layout();save(fig,'05_novelty_and_fallback')
display(table);display(pd.DataFrame(feature_counts))

## 6. 상위 표현의 보존과 학습 근거 · cold-site 준비
GPT 검토를 반영해 미지 exact에서 유전자·유형·아미노산 정보가 남는 비율과 **학습 부분에서 실제 관측된 비율**을 나눕니다. AA는 전역 치환 쌍 외에 변화 전·후 개별 아미노산도 공유합니다. 미지와 희귀 제외를 구분하고 변이가 없는 환자의 토큰 OOV 비율은 N/A입니다.

추가로 전체 train에서 빈번한 위치 하나를 고르고 그 위치를 가진 모든 canonical group을 별도 평가군으로 제외합니다. 이 cold-site는 전체 OOF의 대체가 아닌 제한된 신규 위치 스트레스 시나리오입니다. 모든 학습 클래스가 남는지와 평가군의 실제 클래스 구성을 공개하며 분류기는 학습하지 않습니다.

In [ ]:
coverage_rows=[]
blocks=[('gene','novel_tokens','novel_gene_preserved','novel_gene_supported'),('type','novel_parsed_type_n','novel_type_preserved','novel_type_supported'),('AA pair','novel_aa_n','novel_aa_pair_preserved','novel_aa_pair_supported'),('AA from/to','novel_aa_n','novel_aa_components_preserved','novel_aa_components_supported')]
for fold,z in novel_all.groupby('fold'):
    for name,den_col,ret_col,sup_col in blocks:
        den=int(z[den_col].sum());ret=int(z[ret_col].sum());sup=int(z[sup_col].sum())
        coverage_rows.append({'fold':int(fold),'block':name,'eligible_novel_tokens':den,'preserved':ret,'observed_in_training':sup,'preservation_rate':ret/den if den else np.nan,'training_support_rate':sup/den if den else np.nan})
coverage=pd.DataFrame(coverage_rows);coverage.to_csv(OUT/'novel_upper_feature_support.csv',index=False)
fig,(ax,bx)=plt.subplots(1,2,figsize=(14,5.8))
for a,col,title in [(ax,'preservation_rate','変換後に残る表現'.replace('変換後に残る表現','변환 후 남는 표현')),(bx,'training_support_rate','학습 부분에서 실제 관측된 표현')]:
    z=coverage.pivot(index='fold',columns='block',values=col).reindex(columns=[b[0] for b in blocks])
    im=heat(a,z,z.columns,[f'fold {i}' for i in z.index],[[f'{v:.1%}' if pd.notna(v) else 'N/A' for v in row] for row in z.values],vmax=1)
    a.set_title(title,pad=14)
fig.suptitle('앵커 EDA 6 | 미지 변이의 표현 보존과 학습 근거',fontsize=17,y=1.02)
fig.text(.02,-.035,SCOPE+' · 분모=각 블록을 해석할 수 있는 미지 exact 표기 수 · 표기 자체의 새 데이터 정확도는 미측정',fontsize=10)
fig.tight_layout();save(fig,'06_upper_preservation_and_support');display(coverage)
patient_novel_summary=novel_all.groupby('fold').agg(patients=('ID','size'),no_variant=('exact_n',lambda z:int((z==0).sum())),any_unseen_exact=('exact_unseen',lambda z:int((z>0).sum())),any_unseen_site=('site_unseen',lambda z:int((z>0).sum())),unsupported_upper=('novel_no_supported_upper',lambda z:int((z>0).sum())))
patient_novel_summary.to_csv(OUT/'novelty_patient_denominators.csv')
site_profiles={}
for s in samples:
    for site in s['site']:site_profiles.setdefault(site,set()).add(s['canonical_hash'])
chosen=None
for site,gs in sorted(site_profiles.items(),key=lambda item:(-len(item[1]),item[0])):
    if not 20<=len(gs)<=len(set(groups))*.15:continue
    va=np.flatnonzero(np.isin(groups,list(gs)));tr=np.flatnonzero(~np.isin(groups,list(gs)))
    if set(y[tr])==set(range(len(class_order))) and len(set(y[va]))>=2:
        chosen=(site,gs,tr,va);break
assert chosen is not None,'No feasible cold-site scenario; do not declare full preparation complete'
site,gs,tr,va=chosen
cold_enc=AnchorEncoder(genes,stage_genes,min_support=3).fit([samples[i] for i in tr])
assert site not in cold_enc.seen['site'] and set(groups[tr]).isdisjoint(groups[va])
cold_xtr=cold_enc.transform([samples[i] for i in tr]);cold_xva=cold_enc.transform([samples[i] for i in va])
assert cold_xtr.shape[1]==cold_xva.shape[1]
save_npz(OUT/'cold_train.npz',cold_xtr);save_npz(OUT/'cold_valid.npz',cold_xva)
with gzip.open(OUT/'cold_encoder.json.gz','wt') as f:json.dump(cold_enc.to_dict(),f)
cold_assign=assignments[['ID','SUBCLASS','canonical_hash']].copy();cold_assign['split']='train';cold_assign.loc[va,'split']='cold_validation';cold_assign.to_csv(OUT/'cold_site_split.csv',index=False)
cold_support=cold_assign.groupby(['split','SUBCLASS']).agg(patients=('ID','size'),canonical_groups=('canonical_hash','nunique')).reset_index();cold_support.to_csv(OUT/'cold_site_class_support.csv',index=False)
cold_novel=cold_enc.novelty([samples[i] for i in va]);cold_novel.to_csv(OUT/'cold_site_novelty.csv',index=False)
cold_report={'site':site,'train_rows':len(tr),'validation_rows':len(va),'heldout_profiles':len(gs),'group_overlap':0,'held_site_seen_in_training':False,'all_train_classes_present':True,'validation_classes':sorted(train.SUBCLASS.iloc[va].unique()),'selection':'highest distinct-profile frequency within 20..15% profiles, retaining all training classes; no model score selection','model_trained':False}
(OUT/'cold_site_plan.json').write_text(json.dumps(cold_report,indent=2));display(cold_support);print('Cold-site ready:',cold_report,flush=True)

## 7. 학습 시작 전 준비 결과
첫 앵커 후보는 **L2 다중분류 로지스틱 회귀 + 위 공통 입력**입니다. 모든 블록을 함께 담되 구조가 단순해 다음 변경의 기준점으로 삼기 쉽습니다. 위치·유전자 조합의 비선형 상호작용을 자동으로 모두 학습하는 모델은 아니며 최고 성능을 보장하지 않습니다. 기존 ordinal XGB는 역사적 참고 실험으로 유지합니다.

설정은 결과를 보고 조정하지 않은 최초 고정값입니다. GPT의 최소 지원도 2 제안에 대해서는 이번 사전 초안의 3을 유지합니다(어느 값도 최적값이라는 근거 없음). 전역 요약은 학습 최대값 대신 유전자 스키마의 이론적 분모로 0~1에 고정해 추가 추정 없이 스케일을 맞춥니다. 실제 학습 후 수렴·클래스별 F1·OOF 확률·미지 표기 하위군 성능을 확인해야 앵커로 채택할 수 있습니다. 환자/기관/시간/참조 전사체 정보가 없으므로 새 병원·새 코호트 적응은 별도 데이터 없이는 입증할 수 없습니다.

In [ ]:
config={'anchor_id':'cbj-unified-anchor-v1','representation_version':VERSION,'model':'L2 multinomial LogisticRegression','model_params':{'solver':'saga','C':1.0,'penalty':'l2','max_iter':2000,'tol':0.001,'random_state':42,'class_weight':None},'n_folds':5,'fold_seed':42,'min_support_distinct_training_profiles':3,'class_order':class_order,'source_genes':genes,'stage_genes':stage_genes,'training_enabled':False,'feature_weights':'all 1; no privileged pathway weighting','matrix_values':'binary or defined fractions in [0,1]','primary_metric':'OOF macro F1 over all 26 classes','new_data_claim':'representation fallback verified; predictive improvement NOT VERIFIED'}
(OUT/'anchor_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2))
# Synthetic contract cases are tests only, never part of the EDA data/plots.
probe=pd.DataFrame({'ID':['synthetic_unknown'],'IDH1':['R999999C']})
probe=probe.reindex(columns=['ID']+genes,fill_value='WT')
probe_samples,_,_=build_samples(probe,genes,stage_genes);probe_x=full_enc.transform(probe_samples)
assert probe_x[0,full_enc.index['gene|IDH1|nonWT']]==1 and probe_x[0,full_enc.index['type|IDH1|missense']]==1
assert probe_x[0,full_enc.index['aa|R>C']]>0
missing_probe=probe.drop(columns='IDH1');ps,_,_=build_samples(missing_probe,genes,stage_genes)
assert ps[0]['features']['gene|IDH1|missing']==1
ann.to_csv(OUT/'all_variant_annotations.csv.gz',index=False,compression='gzip');miss.to_csv(OUT/'missing_train_cells.csv',index=False);test_miss.to_csv(OUT/'missing_test_cells.csv',index=False)
patients.to_csv(OUT/'patient_summary.csv',index=False)
parser_counts=ann.groupby(['type','parse_status']).size().rename('fragments').reset_index();parser_counts.to_csv(OUT/'parser_audit.csv',index=False)
ann.loc[ann.parse_status.isin(['unresolved','type_only_ambiguous_position','range_notation_not_synonymous_assumed'])].head(100).to_csv(OUT/'uncertain_notation_examples.csv',index=False)
prepared={'ready_for_training':True,'model_trained':False,'source_rows':{k:len(v) for k,v in frames.items()},'classes':len(class_order),'genes':len(genes),'raw_fragments':len(ann),'unique_patient_gene_tokens':len(u),'canonical_profiles':patients.canonical_hash.nunique(),'canonical_duplicate_extra_rows':len(train)-patients.canonical_hash.nunique(),'canonical_conflicting_groups':len(conflict),'fold_checks':checks,'unit_tests_passed':test_result.testsRun,'novel_mutation_coarse_fallback':True,'missing_gene_is_not_WT':True,'test_vocabulary_not_fitted':True,'full_feature_count':len(full_enc.names),'runtime':{'python':sys.version,'sklearn':sklearn.__version__},'notes':['All-data EDA is exploratory; grouped OOF is internal validation, not a fresh external test.','No classifier fitting, prediction, or submission in this run.','Schema changes require review; unexpected genes are rejected instead of silently discarded.']}
for f in ['cbj_anchor_features.py','cbj_anchor_train.py','test_cbj_anchor_features.py']:
    import shutil
    shutil.copy2('/kaggle/working/'+f,OUT/f)
prepared['cold_site_prepared']=cold_report
prepared['representation_support_audited']=True
prepared['ready_for_training']=bool(test_result.wasSuccessful() and len(checks)==5 and (foldids>=0).all() and cold_report['all_train_classes_present'] and not cold_report['held_site_seen_in_training'])
prepared['file_sha256']={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.iterdir() if p.suffix in ['.npz','.gz','.py'] or p.name in ['anchor_config.json','fold_assignments.csv','cold_site_split.csv']}
(OUT/'preflight.json').write_text(json.dumps(prepared,ensure_ascii=False,indent=2))
display(Markdown('**READY FOR TRAINING（未学習）**'.replace('（未学習）',' · 새 앵커 모델 미학습')))
display(pd.DataFrame(checks));display(parser_counts)
print(json.dumps({k:v for k,v in prepared.items() if k not in ['file_sha256','runtime','fold_checks']},ensure_ascii=False,indent=2),flush=True)